# Food Delivery Time Prediction System Using Linear Regression

**Academic Data Science Project**  
**Dataset:** Kaggle Food Delivery Time Prediction (*denkuznetz*)

---

## 1. Project Overview & Problem Statement
In food delivery systems (e.g. Swiggy, Zomato, DoorDash), predicting delivery duration is crucial for dispatch scheduling and customer expectations.

### Objective:
Predict `Delivery_Time_min` using **Linear Regression** based on:
- `Distance_km` (road distance in km)
- `Weather` (Clear, Windy, Foggy, Rainy, Snowy)
- `Traffic_Level` (Low, Medium, High)
- `Time_of_Day` (Morning, Afternoon, Evening, Night)
- `Vehicle_Type` (Bike, Scooter, Car)
- `Preparation_Time_min` (kitchen prep time in minutes)
- `Courier_Experience_yrs` (courier experience in years)

`Order_ID` is an identifier and is excluded from training.

## 2. Importing Libraries
Importing standard libraries for data handling, visualization, and machine learning.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries successfully imported!")

## 3. Loading Dataset & Inspection
We load `Food_Delivery_Times.csv` and inspect columns, types, and summary statistics.

In [ ]:
data_path = os.path.join("..", "dataset", "Food_Delivery_Times.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 4. Data Cleaning & Preprocessing
1. Check duplicates and remove if found.
2. Drop `Order_ID` column.
3. Check missing values (`NaN`).
4. Impute `Courier_Experience_yrs` with median.
5. Impute `Weather`, `Traffic_Level`, and `Time_of_Day` with mode.

In [ ]:
# Check duplicates
print("Duplicate rows:", df.duplicated().sum())
if df.duplicated().sum() > 0:
    df = df.drop_duplicates()

# Drop Order_ID
if 'Order_ID' in df.columns:
    df = df.drop(columns=['Order_ID'])

# Check missing values
print("\nMissing values before cleaning:")
print(df.isnull().sum())

In [ ]:
# Imputation
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median())
for c in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df[c] = df[c].fillna(df[c].mode()[0])

print("Missing values after cleaning:")
print(df.isnull().sum())

## 5. Exploratory Data Analysis (EDA)
Visualizing distributions and relationships between delivery time and predictors.

In [ ]:
# 5.1 Distribution of Delivery Times
plt.figure(figsize=(9, 5))
sns.histplot(df['Delivery_Time_min'], kde=True, color="#1e3a8a", bins=30)
plt.title("Distribution of Delivery Times (Minutes)", fontsize=14, weight="bold")
plt.xlabel("Delivery Time (min)")
plt.ylabel("Count")
plt.axvline(df['Delivery_Time_min'].mean(), color="red", linestyle="--", label=f"Mean: {df['Delivery_Time_min'].mean():.1f} min")
plt.axvline(df['Delivery_Time_min'].median(), color="green", linestyle=":", label=f"Median: {df['Delivery_Time_min'].median():.1f} min")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Distance vs Delivery Time
plt.figure(figsize=(9, 5))
sns.regplot(data=df, x="Distance_km", y="Delivery_Time_min",
            scatter_kws={"alpha": 0.4, "color": "#2563eb"},
            line_kws={"color": "#dc2626", "linewidth": 2.5})
plt.title("Delivery Distance vs Delivery Time", fontsize=14, weight="bold")
plt.xlabel("Distance (km)")
plt.ylabel("Delivery Time (min)")
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Traffic Level vs Delivery Time
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Traffic_Level", y="Delivery_Time_min", order=["Low", "Medium", "High"], palette="Blues")
plt.title("Traffic Level vs Delivery Time", fontsize=14, weight="bold")
plt.xlabel("Traffic Level")
plt.ylabel("Delivery Time (min)")
plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Weather Condition vs Delivery Time
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="Weather", y="Delivery_Time_min", order=["Clear", "Windy", "Foggy", "Rainy", "Snowy"], palette="Reds", errorbar=None)
plt.title("Average Delivery Time by Weather Condition", fontsize=14, weight="bold")
plt.xlabel("Weather Condition")
plt.ylabel("Mean Delivery Time (min)")
plt.tight_layout()
plt.show()

In [ ]:
# 5.5 Preparation Time vs Delivery Time
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="Preparation_Time_min", y="Delivery_Time_min", hue="Vehicle_Type", alpha=0.6, palette="Set2")
plt.title("Preparation Time vs Delivery Time (by Vehicle Type)", fontsize=14, weight="bold")
plt.xlabel("Preparation Time (min)")
plt.ylabel("Delivery Time (min)")
plt.tight_layout()
plt.show()

In [ ]:
# 5.6 Pearson Correlation Heatmap
num_cols = ["Distance_km", "Preparation_Time_min", "Courier_Experience_yrs", "Delivery_Time_min"]
corr = df[num_cols].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".3f", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 6. Simple Linear Regression (Baseline Model)
Models $y = mx + c$ using `Distance_km` only.

In [ ]:
X_simple = df[["Distance_km"]]
y = df["Delivery_Time_min"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_simple, y, test_size=0.2, random_state=42)

slr = LinearRegression()
slr.fit(X_train_s, y_train_s)
y_pred_s = slr.predict(X_test_s)

mae_s = mean_absolute_error(y_test_s, y_pred_s)
mse_s = mean_squared_error(y_test_s, y_pred_s)
rmse_s = np.sqrt(mse_s)
r2_s = r2_score(y_test_s, y_pred_s)

print(f"Formula: Delivery_Time = {slr.coef_[0]:.4f} * Distance_km + {slr.intercept_:.4f}")
print(f"MAE:  {mae_s:.2f} min")
print(f"MSE:  {mse_s:.2f}")
print(f"RMSE: {rmse_s:.2f} min")
print(f"R2 Score: {r2_s:.4f} ({r2_s*100:.1f}%)")

In [ ]:
# Plot SLR Line
plt.figure(figsize=(9, 5))
plt.scatter(X_test_s, y_test_s, color="#3b528b", alpha=0.5, label="Actual Test Data")
plt.plot(X_test_s, y_pred_s, color="#dc2626", linewidth=2.5, label=f"Regression Line (y = {slr.coef_[0]:.2f}x + {slr.intercept_:.2f})")
plt.title("Simple Linear Regression: Distance vs Delivery Time", fontsize=14, weight="bold")
plt.xlabel("Distance (km)")
plt.ylabel("Delivery Time (min)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Multiple Linear Regression (Primary Model)
Models $Y = \beta_0 + \sum \beta_i X_i$ using all input features.

In [ ]:
num_features = ["Distance_km", "Preparation_Time_min", "Courier_Experience_yrs"]
cat_features = ["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]

X = df[num_features + cat_features]
y = df["Delivery_Time_min"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_features)
    ]
)

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model_pipeline.fit(X_train, y_train)
y_pred_m = model_pipeline.predict(X_test)

mae_m = mean_absolute_error(y_test, y_pred_m)
mse_m = mean_squared_error(y_test, y_pred_m)
rmse_m = np.sqrt(mse_m)
r2_m = r2_score(y_test, y_pred_m)

print("Multiple Linear Regression Results:")
print(f"MAE:  {mae_m:.2f} min")
print(f"MSE:  {mse_m:.2f}")
print(f"RMSE: {rmse_m:.2f} min")
print(f"R2 Score: {r2_m:.4f} ({r2_m*100:.1f}%)")

In [ ]:
# Model Comparison Table
pd.DataFrame({
    "Metric": ["MAE (min)", "MSE", "RMSE (min)", "R2 Score"],
    "Simple Linear Regression": [f"{mae_s:.2f}", f"{mse_s:.2f}", f"{rmse_s:.2f}", f"{r2_s:.4f}"],
    "Multiple Linear Regression": [f"{mae_m:.2f}", f"{mse_m:.2f}", f"{rmse_m:.2f}", f"{r2_m:.4f}"]
})

In [ ]:
# 7.1 Actual vs. Predicted Delivery Time
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred_m, color="#1e3a8a", alpha=0.6, edgecolors="w", s=60)
min_v, max_v = min(y_test.min(), y_pred_m.min()), max(y_test.max(), y_pred_m.max())
plt.plot([min_v, max_v], [min_v, max_v], color="#dc2626", linestyle="--", linewidth=2, label="Ideal (y = x)")
plt.title("Multiple Linear Regression: Actual vs Predicted", fontsize=14, weight="bold")
plt.xlabel("Actual Delivery Time (min)")
plt.ylabel("Predicted Delivery Time (min)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 Learned Feature Coefficients
cat_encoder = model_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
cat_names = list(cat_encoder.get_feature_names_out(cat_features))
all_names = num_features + cat_names
reg = model_pipeline.named_steps["regressor"]

coef_df = pd.DataFrame({"Feature": all_names, "Coefficient": reg.coef_}).sort_values(by="Coefficient")

plt.figure(figsize=(10, 6))
colors = ["#dc2626" if c > 0 else "#2563eb" for c in coef_df["Coefficient"]]
plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("Feature Coefficients Impact (Minutes)", fontsize=14, weight="bold")
plt.xlabel("Coefficient (min)")
plt.tight_layout()
plt.show()

## 8. Summary & Key Findings
- Multiple Linear Regression improves $R^2$ from 0.647 to 0.826 and reduces MAE from 9.58 min to 5.90 min.
- Distance contributes ~3.00 minutes per km.
- Kitchen preparation time adds ~0.97 minutes per minute.
- Courier experience saves ~0.68 minutes per year of experience.
- Adverse weather (Snowy/Foggy/Rainy) introduces substantial delays (4.6 to 9.2 minutes).